In [17]:
import subprocess

result = subprocess.run(
    ["obabel", "-L", "formats"],
    capture_output=True, text=True
)

output = result.stdout + result.stderr

# Search for pdbqt manually
found = False
for line in output.split('\n'):
    if 'pdbqt' in line.lower():
        print(f"✅ pdbqt found: {line}")
        found = True

if not found:
    print("❌ pdbqt still not found")
    print("\nAll available formats containing 'pdb':")
    for line in output.split('\n'):
        if 'pdb' in line.lower():
            print(f"  {line}")

✅ pdbqt found: pdbqt -- AutoDock PDBQT format


In [19]:
print("Converting receptor to pdbqt...")

result = subprocess.run([
    "obabel",
    "-ipdb", PROTEIN_PDB,
    "-opdbqt",
    "-O", "docking/receptor.pdbqt",
    "--addh",
    "-p", "7.4",
    "-xr"
], capture_output=True, text=True)

print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)

if os.path.exists("docking/receptor.pdbqt"):
    size = os.path.getsize("docking/receptor.pdbqt")
    print(f"✅ receptor.pdbqt created — {size} bytes")
    print("\nFirst 5 lines:")
    with open("docking/receptor.pdbqt") as f:
        for i, line in enumerate(f):
            if i >= 5: break
            print(f"  {line.rstrip()}")
else:
    print("❌ Failed")

Converting receptor to pdbqt...
STDOUT: 
STDERR: 1 molecule converted

✅ receptor.pdbqt created — 57113 bytes

First 5 lines:
  REMARK  Name = data/clean_protein.pdb
  REMARK                            x       y       z     vdW  Elec       q    Type
  REMARK                         _______ _______ _______ _____ _____    ______ ____
  ATOM      1  CA  TYR A   6      -7.551 -11.355 -17.946  0.00  0.00    +0.000 C
  TER


In [11]:
import subprocess, os, json

# ============================================================
# SETTINGS
# ============================================================
VINA_PATH      = r"D:\mdr-pepdesign\tools\vina_1.2.7_win.exe"
OBABEL_PATH    = "obabel"        # conda version — works globally
PROTEIN_NAME   = "EmrE"
PROTEIN_PDB    = "data/clean_protein.pdb"
COORDS_FILE    = "data/pocket_coords.txt"
GRID_SIZE      = (25, 25, 25)
EXHAUSTIVENESS = 8
NUM_MODES      = 9
# ============================================================

os.chdir(r"D:\mdr-pepdesign")
os.makedirs("docking",         exist_ok=True)
os.makedirs("docking/ligands", exist_ok=True)
os.makedirs("docking/results", exist_ok=True)

print("✅ Settings ready")

✅ Settings loaded
   Vina            : True
   MGLTools Python : True
   Receptor script : True
   Ligand script   : True
   Protein PDB     : True
   Coords file     : True


In [20]:
print("Reading pocket coordinates...")

with open(COORDS_FILE) as f:
    coords = json.load(f)

cx = coords['cx']
cy = coords['cy']
cz = coords['cz']

print(f"Pocket centre: X={cx}, Y={cy}, Z={cz}")

config = f"""receptor = docking/receptor.pdbqt
center_x = {cx}
center_y = {cy}
center_z = {cz}
size_x = {GRID_SIZE[0]}
size_y = {GRID_SIZE[1]}
size_z = {GRID_SIZE[2]}
exhaustiveness = {EXHAUSTIVENESS}
num_modes = {NUM_MODES}
energy_range = 3"""

with open("docking/vina.conf", "w") as f:
    f.write(config)

print("✅ vina.conf written")
print()
print(config)

Reading pocket coordinates...
Pocket centre: X=2.733, Y=1.033, Z=-13.822
✅ vina.conf written

receptor = docking/receptor.pdbqt
center_x = 2.733
center_y = 1.033
center_z = -13.822
size_x = 25
size_y = 25
size_z = 25
exhaustiveness = 8
num_modes = 9
energy_range = 3


In [22]:
result = subprocess.run(
    [VINA_PATH, "--version"],
    capture_output=True, text=True
)

print(result.stdout.strip())
print("✅ Vina ready")

AutoDock Vina v1.2.7
✅ Vina ready


In [21]:
def convert_ligand(pdb_file, out_pdbqt):
    """
    Convert any peptide PDB to PDBQT.
    Generic — works for any protein or peptide.
    """
    result = subprocess.run([
        OBABEL_PATH,
        "-ipdb", pdb_file,
        "-opdbqt",
        "-O", out_pdbqt,
        "--addh",
        "-p", "7.4",
        "-xh"        # flexible bonds for ligand mode
    ], capture_output=True, text=True)
    return os.path.exists(out_pdbqt)

print("✅ Ligand conversion function ready")

✅ Ligand conversion function ready
